In [1]:
#kernel thesis clean4
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_pickle("C:\\Users\\Patrick\\Masterthesis\\Benchmarks\\screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [3]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [4]:
import torch
import torch.nn as nn
import sys
sys.path.append(r"C:\Users\Patrick\InceptionTime-Pytorch")
from inception import InceptionBlock


class Flatten(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x.mean(-1)   # safer als view


model = nn.Sequential(
    InceptionBlock(
        in_channels=2,  # 2 Features: torque and E_kin
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)

# cv Momentanleistung der Rotation S.136 kuchling

In [ ]:
# P = M* w ; w = diff(angle/time)
import numpy as np

torque = np.array(df['torque_values'].tolist())[..., np.newaxis]
angle = np.array(df['angle_values'].tolist())
time  = np.array(df['time_values'].tolist())

angle_rad = np.radians(angle)

dt = np.diff(time, axis=1) + 1e-8
dangle = np.diff(angle_rad, axis=1)

w = dangle /dt  


torque_trim = torque[:, 1:, 0] 

#momentanleistung
power = torque_trim * w   
power = power[..., np.newaxis]  

print( power.shape)
print( torque_trim.shape)

(12500, 799, 1)
(12500, 799)


In [8]:
from sklearn.metrics import f1_score
import math
torque = np.array(df['torque_values'].tolist())
angle = np.array(df['angle_values'].tolist())
time  = np.array(df['time_values'].tolist())
phase = np.array(df['step_values'].tolist())

torque = np.array(df['torque_values'].tolist())[..., np.newaxis]
angle = np.array(df['angle_values'].tolist())
time  = np.array(df['time_values'].tolist())

angle_rad = np.radians(angle)

dt = np.diff(time, axis=1) + 1e-8
dangle = np.diff(angle_rad, axis=1)

w = dangle /dt  


torque_trim = torque[:, 1:, 0] 

#momentanleistung
power = torque_trim * w   
power = power[..., np.newaxis]  

torque_trim_3d = torque_trim[..., np.newaxis]

x_data = np.concatenate([torque_trim_3d, power], axis=-1)
y_data = np.array(df['class_values'].tolist())
print("x_data shape:", x_data.shape)
x_data = np.transpose(x_data, (0, 2, 1))
print("transposed x_data shape:", x_data.shape)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=81)

cv_scores = []

best_overall_model_state = None
best_overall_f1 = -np.inf


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"Fold {fold+1}")

    X_train_fold = X_train_full[train_idx]
    y_train_fold = y_train_full[train_idx]

    X_val_fold = X_train_full[val_idx]
    y_val_fold = y_train_full[val_idx]

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32),torch.tensor(y_train_fold, dtype=torch.long)),batch_size=32,shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32),torch.tensor(y_val_fold, dtype=torch.long)),batch_size=32,shuffle=False)
    model = nn.Sequential(
    InceptionBlock(
        in_channels=2,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    earlystop = EarlyStopper(patience=7, min_delta=0.001)

    epochs = 50

    best_val_f1 = -np.inf
    best_model_state = None


    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                X_val_batch = X_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                outputs = model(X_val_batch)
                loss = criterion(outputs, y_val_batch)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_val_batch.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        scheduler.step(avg_val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()

        if earlystop.early_stop(avg_val_loss):
            print("Early stopping triggered")
            break

        print(f"Fold {fold+1}, Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")

    cv_scores.append(best_val_f1)
    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_model_state = best_model_state


print(f"CV f1 mean avg score: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")

x_data shape: (12500, 799, 2)
transposed x_data shape: (12500, 2, 799)
Fold 1
Fold 1, Epoch 1/50, Train Loss: 1.8277, Val Loss: 1.9979, Val F1: 0.1500
Fold 1, Epoch 2/50, Train Loss: 1.7104, Val Loss: 1.8495, Val F1: 0.2015
Fold 1, Epoch 3/50, Train Loss: 1.6790, Val Loss: 1.9357, Val F1: 0.1818
Fold 1, Epoch 4/50, Train Loss: 1.6540, Val Loss: 1.6383, Val F1: 0.2379
Fold 1, Epoch 5/50, Train Loss: 1.6326, Val Loss: 2.3627, Val F1: 0.1568
Fold 1, Epoch 6/50, Train Loss: 1.5906, Val Loss: 7.8874, Val F1: 0.0489
Fold 1, Epoch 7/50, Train Loss: 1.5616, Val Loss: 1.5595, Val F1: 0.2810
Fold 1, Epoch 8/50, Train Loss: 1.5444, Val Loss: 1.8463, Val F1: 0.2280
Fold 1, Epoch 9/50, Train Loss: 1.5263, Val Loss: 1.5214, Val F1: 0.3334
Fold 1, Epoch 10/50, Train Loss: 1.5287, Val Loss: 1.8706, Val F1: 0.2276
Fold 1, Epoch 11/50, Train Loss: 1.4966, Val Loss: 1.9501, Val F1: 0.2248
Fold 1, Epoch 12/50, Train Loss: 1.5048, Val Loss: 1.8537, Val F1: 0.2166
Fold 1, Epoch 13/50, Train Loss: 1.4912, Va

In [9]:
model.load_state_dict(best_model_state)
#torch.save(best_model_state, "best_inception_model.pth")
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),torch.tensor(y_test, dtype=torch.long)),batch_size=32,shuffle=False)
with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 1.2222
Test F1 Macro: 0.4276
